# 04 — Evaluation
Compare all models on the held-out test set using PR-AUC, F1, precision, recall.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import pickle
from src.evaluate import evaluate_model, plot_pr_curves, plot_confusion_matrices

X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()
print(f'Test set: {X_test.shape}, fraud rate: {y_test.mean():.4f}')

Test set: (56962, 30), fraud rate: 0.0017


In [2]:
models = {}
for name in ['lr', 'rf', 'xgb', 'nn']:
    with open(f'../results/{name}.pkl', 'rb') as f:
        models[name] = pickle.load(f)
print('Loaded models:', list(models.keys()))

FileNotFoundError: [Errno 2] No such file or directory: '../results/xgb.pkl'

In [ ]:
labels = {
    'lr':  'Logistic Regression',
    'rf':  'Random Forest',
    'xgb': 'XGBoost',
    'nn':  'Neural Network',
}

results = []
for key, model in models.items():
    r = evaluate_model(labels[key], model, X_test, y_test)
    results.append(r)

In [ ]:
plot_pr_curves(results, y_test, save_path='../results/figures/pr_curves.png')

In [ ]:
plot_confusion_matrices(results, y_test, save_path='../results/figures/confusion_matrices.png')

In [ ]:
# Summary table
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

rows = []
for r in results:
    rows.append({
        'Model': r['name'],
        'PR-AUC': round(r['pr_auc'], 4),
        'Precision': round(precision_score(y_test, r['preds']), 4),
        'Recall': round(recall_score(y_test, r['preds']), 4),
        'F1': round(f1_score(y_test, r['preds']), 4),
    })

summary = pd.DataFrame(rows).sort_values('PR-AUC', ascending=False)
print(summary.to_string(index=False))
summary.to_csv('../results/model_comparison.csv', index=False)